# Model Explainability

**Topic:** Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox
from IPython.display import display, HTML, clear_output
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout, plot_feature_importance

try:
    from sklearn.datasets import fetch_openml
    _hd = fetch_openml(name='heart-disease', version=1, as_frame=True)
    X = _hd.data
    y = (_hd.target.astype(int) > 0).astype(int)
except Exception:
    from sklearn.datasets import load_breast_cancer as _lbc
    _bc = _lbc(as_frame=True)
    X, y = _bc.data, _bc.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
## What you'll explore

By the end of this session you will be able to:

- **Explain** the difference between global feature importance and local prediction explanation
- **Interpret** a SHAP summary plot to understand which features drive model predictions overall
- **Describe** what LIME does differently from SHAP and when each approach is more appropriate

> **Tip:** Global feature importance tells you which features matter *on average*. Local explanation tells you which features drove *this specific prediction*. For a medical model, the local explanation is almost always more useful to the clinician.

---
## How we got here

In `ml_concepts/13_interpretability_vs_complexity.ipynb` you placed each algorithm on the interpretability-complexity spectrum. Decision trees and linear models are interpretable by design; Random Forests and neural networks are not.

SHAP (SHapley Additive exPlanations) and LIME (Local Interpretable Model-agnostic Explanations) are post-hoc methods that apply to *any* algorithm after training — they let you peer inside black box models without changing the model architecture. GDPR's "right to explanation" for automated decisions is one of the key regulatory drivers behind these tools.

---
## Why this matters for data science

Two reasons to explain predictions: one technical, one ethical.

Technically, unexplained models fail silently. If a house price model has learned that a specific census tract identifier is its strongest predictor — a proxy for redlining — no accuracy metric will catch that. Explainability tools surface these proxy features before deployment.

Ethically, in high-stakes domains (medicine, criminal justice, lending), a model must be auditable. A doctor who overrides a clinical decision support tool needs to understand why the model flagged this patient — not just that it did.

---
## Try it yourself

In [ ]:
import shap

out1 = Output()
caption1 = widgets.HTML()

rf_explain = RandomForestClassifier(n_estimators=100, random_state=42)
rf_explain.fit(X_train, y_train)
shap_explainer = shap.TreeExplainer(rf_explain)
shap_exp = shap_explainer(X_test)

feature_names = list(X_test.columns)
n_show = min(20, len(X_test))
pred_probs = rf_explain.predict_proba(X_test.iloc[:n_show])[:, 1]
class_labels = ["No disease", "Disease"]
patient_options = [
    (f"Patient {i} — true: {class_labels[int(y_test.iloc[i])]}, "
     f"predicted P(disease)={pred_probs[i]:.2f}", i)
    for i in range(n_show)
]

patient_dropdown = Dropdown(
    options=patient_options, value=0,
    description="Patient:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="480px"),
)

def render1(change=None):
    idx = patient_dropdown.value
    shap_vals = shap_exp.values[idx, :, 1]
    base_val = float(shap_exp.base_values[idx, 1])
    feature_vals = X_test.iloc[idx].values
    pred = rf_explain.predict_proba(X_test.iloc[[idx]])[0, 1]

    order = np.argsort(-np.abs(shap_vals))
    top_n = 6
    top_idx = order[:top_n]
    rest_idx = order[top_n:]
    labels = [f"{feature_names[i]} = {feature_vals[i]:.2f}" for i in top_idx]
    values = [float(shap_vals[i]) for i in top_idx]
    if len(rest_idx) > 0:
        labels.append(f"All other {len(rest_idx)} features")
        values.append(float(shap_vals[rest_idx].sum()))
    labels = ["Base rate"] + labels + ["Prediction"]
    measures = ["absolute"] + ["relative"] * len(values) + ["total"]
    ys = [base_val] + values + [0]

    fig = go.Figure(go.Waterfall(
        x=labels, measure=measures, y=ys,
        increasing=dict(marker_color=PALETTE["secondary"]),
        decreasing=dict(marker_color=PALETTE["primary"]),
        totals=dict(marker_color=PALETTE["accent"]),
        connector=dict(line_color=PALETTE["muted"]),
    ))
    fig.update_layout(**{k: v for k, v in base_layout(
        title=f"SHAP Waterfall — Patient {idx} (predicted P(disease) = {pred:.2f})",
        yaxis_title="P(disease)",
    ).to_plotly_json().items()})
    fig.update_layout(height=440, showlegend=False)

    with out1:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    top_feature = labels[1]
    direction = "raises" if values[0] >= 0 else "lowers"
    caption1.value = (
        f"<b>Patient {idx}:</b> starting from the base rate of {base_val:.2f}, the single "
        f"biggest mover is <b>{top_feature}</b>, which {direction} the predicted risk. Adding "
        f"up every feature's contribution lands on a final predicted P(disease) of <b>{pred:.2f}</b> "
        f"— switch patients to see a completely different set of features doing the driving."
    )

patient_dropdown.observe(render1, names="value")
display(VBox([patient_dropdown, out1, caption1]))
render1()

In [ ]:
import shap
import lime.lime_tabular
from plotly.subplots import make_subplots

out2 = Output()
caption2 = widgets.HTML()

rf_explain2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_explain2.fit(X_train, y_train)
shap_explainer2 = shap.TreeExplainer(rf_explain2)
shap_exp2 = shap_explainer2(X_test)

feature_names2 = list(X_test.columns)
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values, feature_names=feature_names2,
    class_names=["No disease", "Disease"], mode="classification", random_state=42,
)

n_show2 = min(20, len(X_test))
pred_probs2 = rf_explain2.predict_proba(X_test.iloc[:n_show2])[:, 1]
class_labels2 = ["No disease", "Disease"]
patient_options2 = [
    (f"Patient {i} — true: {class_labels2[int(y_test.iloc[i])]}, "
     f"predicted P(disease)={pred_probs2[i]:.2f}", i)
    for i in range(n_show2)
]

patient_dropdown2 = Dropdown(
    options=patient_options2, value=0,
    description="Patient:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="480px"),
)

def lime_feature_name(condition, names):
    for name in sorted(names, key=len, reverse=True):
        if name in condition:
            return name
    return condition

def render2(change=None):
    idx = patient_dropdown2.value
    shap_vals = shap_exp2.values[idx, :, 1]
    top_n = 6
    order = np.argsort(-np.abs(shap_vals))[:top_n]
    shap_labels = [feature_names2[i] for i in order]
    shap_values_top = [float(shap_vals[i]) for i in order]

    lime_exp = lime_explainer.explain_instance(
        X_test.iloc[idx].values, rf_explain2.predict_proba, num_features=top_n,
    )
    lime_pairs = lime_exp.as_list()
    lime_labels = [lime_feature_name(p[0], feature_names2) for p in lime_pairs]
    lime_values = [p[1] for p in lime_pairs]

    fig = make_subplots(rows=1, cols=2,
                         subplot_titles=("SHAP contributions", "LIME contributions"))
    fig.add_trace(go.Bar(
        x=shap_values_top, y=shap_labels, orientation="h",
        marker_color=[PALETTE["secondary"] if v >= 0 else PALETTE["primary"]
                      for v in shap_values_top],
        showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=lime_values, y=lime_labels, orientation="h",
        marker_color=[PALETTE["secondary"] if v >= 0 else PALETTE["primary"]
                      for v in lime_values],
        showlegend=False,
    ), row=1, col=2)

    fig.update_layout(**{k: v for k, v in base_layout(
        title=f"SHAP vs. LIME — Patient {idx}",
    ).to_plotly_json().items()})
    fig.update_layout(height=420, margin=dict(l=90, r=40, t=80, b=40))

    with out2:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    shap_top_feat = shap_labels[0]
    lime_top_feat = lime_labels[0]
    agree_note = (
        "SHAP and LIME agree on the top feature here"
        if shap_top_feat == lime_top_feat else
        f"SHAP's top feature ({shap_top_feat}) differs from LIME's top feature "
        f"({lime_top_feat}) for this patient"
    )
    caption2.value = (
        f"<b>Patient {idx}:</b> {agree_note}. SHAP values are computed exactly from the "
        f"trained forest and stay consistent every time you re-run this patient. LIME "
        f"re-fits a small local linear model on randomly perturbed samples each call, so "
        f"its bars can shift slightly between runs even for the same patient."
    )

patient_dropdown2.observe(render2, names="value")
display(VBox([patient_dropdown2, out2, caption2]))
render2()

---
## What's happening?

**Feature importance (built-in)** measures how much each feature reduces impurity across all splits in a tree ensemble. It is global and fast but biased toward high-cardinality features and does not show direction (positive/negative effect).

**Permutation importance** randomly shuffles one feature at a time and measures how much the validation score drops. It is model-agnostic, unbiased, and works on any fitted model.

**SHAP** (SHapley Additive exPlanations) uses game theory to fairly attribute the prediction to each feature. For each prediction, it computes how much each feature "contributed" by comparing all subsets of features. It is consistent — if a feature contributes more in one model, its SHAP value is always higher.

**LIME** (Local Interpretable Model-agnostic Explanations) fits a simple linear model around a single prediction using perturbed samples. It is faster than SHAP for one-off explanations but can produce inconsistent results across runs.

| Method | How it works | Local or global | Consistent | Speed | Best for |
|---|---|---|---|---|---|
| Built-in importance | Impurity reduction | Global | No (biased) | Fast | Quick baseline |
| Permutation importance | Score drop on shuffle | Global | Yes | Medium | Any model, any metric |
| SHAP | Shapley game values | Both | Yes | Slow | Production, GDPR audits |
| LIME | Local linear approx | Local | No | Fast | One-off ad hoc explanations |

---
## Real-world example: Explaining why the model flagged this patient

A Random Forest trained on Heart Disease data achieves strong classification performance. But before deploying it in a clinical tool, we need to understand what the model has learned. Feature importance and permutation importance show which features drive predictions on average — a first audit before implementing full SHAP explanations.

In [4]:
import numpy as np
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout, plot_feature_importance

np.random.seed(42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Built-in feature importance
importances = rf.feature_importances_
feature_names = list(X.columns)

fig1 = plot_feature_importance(
    feature_names, importances,
    title="Random Forest — Built-in Feature Importance (Heart Disease)",
    top_n=min(15, len(feature_names)),
)
fig1.show()

# Permutation importance on test set
perm = permutation_importance(rf, X_test, y_test,
    n_repeats=10, random_state=42, scoring='f1')
perm_means = perm.importances_mean

fig2 = plot_feature_importance(
    feature_names, perm_means,
    title="Permutation Importance on Test Set — Heart Disease",
    top_n=min(15, len(feature_names)),
)
fig2.show()

---
## Key takeaway

> **Global feature importance tells you what the model learned on average; local explanation tools like SHAP tell you why it made this specific prediction — and that local story is what matters in high-stakes applications.**

---
*Next up: `10_ab_testing_for_models.ipynb` — applying hypothesis testing to answer whether model B is actually better than model A*